#### 1. embedding layer

In [3]:
import torch
from transformers import AutoTokenizer, AutoModel

In [4]:
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B')
model = AutoModel.from_pretrained('Qwen/Qwen2.5-0.5B')

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [5]:
sentence = "He ate it all"

In [6]:
inputs = tokenizer(sentence, return_tensors="pt")
input_ids = inputs['input_ids']
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

In [7]:
tokens

['He', 'Ġate', 'Ġit', 'Ġall']

In [8]:
with torch.no_grad():
    embeddings = model.embed_tokens(input_ids)

In [9]:
for token, embedding in zip(tokens, embeddings[0]):
    print(f"Token: {token}\n")
    print(f"Embedding: {embedding}\n")

Token: He

Embedding: tensor([-3.1494e-02, -5.1880e-03,  2.7618e-03, -1.5198e-02, -9.9487e-03,
        -2.8229e-03, -7.4463e-03, -1.4404e-02, -1.4648e-02,  1.2756e-02,
        -1.4954e-02, -3.9673e-03, -2.7954e-02, -8.0490e-04, -1.6724e-02,
        -1.1353e-02, -3.2349e-03, -2.3804e-02, -7.7515e-03,  6.3477e-03,
         5.4016e-03, -7.0190e-03,  6.3477e-03, -2.1362e-02,  2.4902e-02,
        -3.0212e-03,  1.2756e-02, -1.4114e-03,  1.7944e-02, -2.8564e-02,
         1.6724e-02,  9.3994e-03, -8.5831e-04, -7.3853e-03, -2.6367e-02,
        -3.7231e-03, -4.8218e-03,  1.7822e-02, -1.4526e-02,  3.0518e-02,
         1.4404e-02, -2.2583e-02,  9.1553e-03,  7.4768e-03, -3.1982e-02,
         2.9802e-05, -1.2329e-02, -7.3853e-03, -7.9346e-03, -1.8555e-02,
        -1.6594e-04, -1.1780e-02,  1.4496e-03, -6.8970e-03, -5.4016e-03,
        -5.8594e-03,  7.3853e-03, -3.2501e-03,  8.4839e-03, -2.4567e-03,
        -5.1880e-03,  1.7212e-02, -3.3447e-02, -1.8799e-02,  3.1738e-03,
        -1.3000e-02, -1.6113e

#### 2. self attention

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

input_embeddings = embeddings  # shape: (1, seq_len, 896)
embed_dim = input_embeddings.size(-1)

wQ = nn.Linear(embed_dim, embed_dim, bias=False)
wK = nn.Linear(embed_dim, embed_dim, bias=False)
wV = nn.Linear(embed_dim, embed_dim, bias=False)

q = wQ(input_embeddings)
k = wK(input_embeddings)
v = wV(input_embeddings)
dim_k = k.size(-1)

attn_scores = torch.matmul(q, k.transpose(-2, -1))
scaled_attn_scores = attn_scores / torch.sqrt(torch.tensor(dim_k, dtype=torch.float32))
normalized_attn_scores = F.softmax(scaled_attn_scores, dim=-1)
output = torch.matmul(normalized_attn_scores, v)

print("input shape :", input_embeddings.shape)
print("attn_scores shape:", attn_scores.shape)
print("output shape     :", output.shape)

input shape : torch.Size([1, 4, 896])
attn_scores shape: torch.Size([1, 4, 4])
output shape     : torch.Size([1, 4, 896])


#### 3. Feed Forward

In [14]:
import torch
import torch.nn as nn

input_dim = embed_dim          # 896
hidden_dim = input_dim * 4     # 3584

class FeedForward(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(FeedForward, self).__init__()
        self.l1 = nn.Linear(input_dim, hidden_dim)
        self.l2 = nn.Linear(hidden_dim, input_dim)
        self.selu = nn.SELU()

    def forward(self, x):
        x = self.selu(self.l1(x))
        x = self.l2(x)
        return x

feed_forward = FeedForward(input_dim, hidden_dim)
outputs = feed_forward(output)  # output: self-attention 결과 (1, 4, 896)
print("output shape:", outputs.shape)


output shape: torch.Size([1, 4, 896])


#### 4. layer normalization

In [15]:
class LayerNorm(nn.Module):
    def __init__(self, dimension, gamma=None, beta=None, epsilon=1e-5):
        super(LayerNorm, self).__init__()
        self.epsilon = epsilon
        self.gamma = gamma if gamma is not None else nn.Parameter(
            torch.ones(dimension)
        )
        self.beta = beta if beta is not None else nn.Parameter(
            torch.zeros(dimension)
        )

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        variance = x.var(-1, keepdim=True, unbiased=False)
        x_normalized = (x - mean) / torch.sqrt(variance + self.epsilon)

        return self.gamma * x_normalized + self.beta

layer_norm = LayerNorm(embed_dim)
outputs = layer_norm(output)
print("output shape:", outputs.shape)


output shape: torch.Size([1, 4, 896])
